# Store Sales Machine Learning
## Machine Learning & Prediction

* **Goal:** Develop a machine learning model to predict sales for individual stores and product families based on historical sales data, promotions, calendar features, and other relevant variables.

In [1]:
# Import libraies
from pathlib import Path
import pandas as pd

data_path = Path('../data/train_ml.csv')
train = pd.read_csv(data_path, parse_dates=['date'])

train.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,year,month,day,day_of_week,is_weekend,quarter,is_special_date,day_of_year
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,1746.0,93.14,2013,1,1,1,0,1,1,1
1,1194,2013-01-01,42,CELEBRATION,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
2,1193,2013-01-01,42,BREAD/BAKERY,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
3,1192,2013-01-01,42,BOOKS,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
4,1191,2013-01-01,42,BEVERAGES,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1


In [2]:
# Inspection
print(f'Rows: {train.shape[0]} -- Cols: {train.shape[1]}')

print(train.columns.tolist())

print(train[['date', 'store_nbr', 'family', 'sales']].head())

Rows: 3000888 -- Cols: 20
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'transactions', 'dcoilwtico', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'quarter', 'is_special_date', 'day_of_year']
        date  store_nbr        family  sales
0 2013-01-01          1    AUTOMOTIVE    0.0
1 2013-01-01         42   CELEBRATION    0.0
2 2013-01-01         42  BREAD/BAKERY    0.0
3 2013-01-01         42         BOOKS    0.0
4 2013-01-01         42     BEVERAGES    0.0


In [3]:
train['date'].min(), train['date'].max()

(Timestamp('2013-01-01 00:00:00'), Timestamp('2017-08-15 00:00:00'))

In [4]:
# Sort data chronologically
train = train.sort_values('date').reset_index(drop=True)

validation_start = '2017-08-01'

# Split data by date
train_data = train[train['date'] < validation_start].copy()
valid_data = train[train['date'] >= validation_start].copy()

# Check split
print('Trainings period:')
print(f' {train_data['date'].min()} to {train_data['date'].max()}')
print('Validation period:')
print(f' {valid_data['date'].min()} to {valid_data['date'].max()}')

Trainings period:
 2013-01-01 00:00:00 to 2017-07-31 00:00:00
Validation period:
 2017-08-01 00:00:00 to 2017-08-15 00:00:00


In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate average sales for each store and product family
sales_mean = (
    train_data
    .groupby(['store_nbr', 'family'])['sales']
    .mean()
)

# Create predition for vald.perido
valid_prediction = valid_data.set_index(
    ['store_nbr', 'family']
).index.map(sales_mean)

# Replace missing pred. with overall training average
valid_prediction = np.where(
    pd.isna(valid_prediction), 
    train_data['sales'].mean(), 
    valid_prediction
)

# Evaluate baseline
mae = mean_absolute_error(valid_data['sales'], valid_prediction)
rmse = np.sqrt(mean_squared_error(valid_data['sales'], valid_prediction))

print(f'Baseline MAE {mae:.2f}')
print(f'Baseline RMSE: {rmse:.2f}')


Baseline MAE 143.64
Baseline RMSE: 506.74


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

In [7]:
# Define target
target = 'sales'

# Define features
categorial_features = ['family', 'city', 'state', 'type']
numeric_features = [
    'store_nbr',
    'cluster',
    'onpromotion',
    'dcoilwtico', 
    'year',
    'month',
    'day',
    'day_of_week',
    'is_weekend', 
    'quarter', 
    'is_special_date', 
    'day_of_year'
]

features = categorial_features + numeric_features

X_train = train_data[features]
y_train = train_data[target]

X_valid = valid_data[features]
y_valid = valid_data[target]

print('Training features:', X_train.shape)
print('Validation features:', X_valid.shape)

Training features: (2974158, 16)
Validation features: (26730, 16)


In [8]:
# Convert categorial feat into numeric values
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorial', 
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            categorial_features
        )
    ],
    remainder='passthrough'
)

# Define machine learning model
model = make_pipeline(
    preprocessor, 
    HistGradientBoostingRegressor(
        max_iter=100, 
        learning_rate=0.1, 
        max_leaf_nodes=31, 
        random_state=42
    )
)

print('Ready')

Ready


In [9]:
# Create sample of df
sample_size = min(300_000, len(X_train))

X_train_sample = X_train.sample(
    n=sample_size, 
    random_state=42
)

y_train_sample = y_train.loc[X_train_sample.index]

print('Sample Size:', len(X_train_sample))

# Train model
model.fit(X_train_sample, y_train_sample)


Sample Size: 300000


,steps,"[('columntransformer', ...), ('histgradientboostingregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorial', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
# Prediction sales
prediction = model.predict(X_valid)

# Evaluate model
mae = mean_absolute_error(y_valid, prediction)
rmse = np.sqrt(mean_squared_error(y_valid, prediction))

print(f'Baseline MAE: 143.64')
print(f'Model MAE: {mae:.2f}')
print(f'Baseline RMSE: 506.74')
print(f'Model RMSE: {rmse:.2f}')

Baseline MAE: 143.64
Model MAE: 145.27
Baseline RMSE: 506.74
Model RMSE: 347.94


In [11]:
# compare actual and predicted sales
results = valid_data[['date', 'store_nbr', 'family', 'sales']].copy()

results['prediction'] = prediction
results['error'] = results['sales'] - results['prediction']
results['absolute_error'] = results['error'].abs()

print('Mean actual sales:', results['sales'].mean())
print('Mean predicted sales:', results['prediction'].mean())
print('Negative prediction:', (results['prediction'] <0).sum())

print('Largest prediction errors:')
display(
    results.sort_values('absolute_error', ascending=False).head(10)
)

Mean actual sales: 465.144891162645
Mean predicted sales: 492.75904559736625
Negative prediction: 3333
Largest prediction errors:


,date,store_nbr,family,sales,prediction,error,absolute_error
2982642,2017-08-05,47,BEVERAGES,14469.000,8184.388256,6284.611744,6284.611744
2984688,2017-08-06,54,BEVERAGES,8552.000,2502.737534,6049.262466,6049.262466
2994893,2017-08-12,40,GROCERY I,10739.000,4989.661244,5749.338756,5749.338756
2980629,2017-08-04,40,BEVERAGES,8272.000,2652.332404,5619.667596,5619.667596
2996214,2017-08-13,28,GROCERY I,11002.664,5675.684742,5326.979258,5326.979258
2994432,2017-08-12,28,GROCERY I,9841.523,4594.782078,5246.740922,5246.740922
2977002,2017-08-02,39,CLEANING,6263.000,1257.385644,5005.614356,5005.614356
2974887,2017-08-01,3,BEVERAGES,8192.000,3306.956271,4885.043729,4885.043729
2996797,2017-08-13,44,GROCERY I,9811.000,14691.143537,-4880.143537,4880.143537
2995026,2017-08-12,44,GROCERY I,9391.000,14138.097977,-4747.097977,4747.097977


In [12]:
# Transform target to reduce influence of large values
y_train_log = np.log1p(y_train_sample)

# Train model with transformed target
model.fit(X_train_sample, y_train_log)

# Predict and convert back to original
prediction_log = model.predict(X_valid)
prediction_log = np.maximum(np.expm1(prediction_log), 0)

mae_log = mean_absolute_error(y_valid, prediction_log)
rmse_log = np.sqrt(mean_squared_error(y_valid, prediction_log))

print(f'Baseline MAE: 143.64')
print(f'Previous model MAE: 145.27')
print(f'Log model MAE: {mae_log:.2f}')

print(f'Baseline RMSE: 506.74')
print(f'Previous model RMSE: 347.94')
print(f'Log model RMSE: {rmse_log:.2f}')


Baseline MAE: 143.64
Previous model MAE: 145.27
Log model MAE: 146.06
Baseline RMSE: 506.74
Previous model RMSE: 347.94
Log model RMSE: 501.06


In [13]:
import catboost
print('Version', catboost.__version__)


Version 1.2.10


In [14]:
from catboost import CatBoostRegressor

# Convert categorial features into strings
X_train_cat = X_train_sample.copy()
X_valid_cat = X_valid.copy()

for col in categorial_features:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_valid_cat[col] = X_valid_cat[col].astype(str)

cat_model = CatBoostRegressor(
    iterations=500, 
    learning_rate=0.1, 
    depth=8, 
    loss_function='RMSE', 
    random_seed=42, 
    verbose=100, 
    allow_writing_files=False
)

cat_model.fit(
    X_train_cat, 
    y_train_sample,
    cat_features=categorial_features
)

0:	learn: 1053.3172361	total: 108ms	remaining: 53.9s
100:	learn: 389.6787132	total: 8.19s	remaining: 32.3s
200:	learn: 346.9695204	total: 16.1s	remaining: 23.9s
300:	learn: 320.5520439	total: 23.8s	remaining: 15.8s
400:	learn: 302.7197609	total: 31.3s	remaining: 7.73s
499:	learn: 290.4486263	total: 39.3s	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=8, iterations=500, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=100)

In [15]:
# Predict sales for validation period
cat_predictions = cat_model.predict(X_valid_cat)

# Evaluation metrics
cat_mae = mean_absolute_error(y_valid, cat_predictions)
cat_rmse = np.sqrt(mean_squared_error(y_valid, cat_predictions))

print('Basline MAE: 143.64')
print('HistGradient MAE: 145.27')
print(f'CatBoost MAE: {cat_mae:.2f}')

print('Basline RMSE: 506.74')
print('HistGradient RMSE: 347.94')
print(f'CatBoost RMSE: {cat_rmse:.2f}')


Basline MAE: 143.64
HistGradient MAE: 145.27
CatBoost MAE: 112.80
Basline RMSE: 506.74
HistGradient RMSE: 347.94
CatBoost RMSE: 303.97


### CatBoost – Model Comparison

CatBoost is a gradient boosting algorithm that builds multiple decision trees sequentially. Each new tree helps reduce the errors made by the previous trees.

**Key feature:** CatBoost can handle categorical features such as `family`, `city`, `state`, and `type` directly. Unlike our HistGradientBoosting model, it does not require manual encoding of these categorical features.

**Validation Results:**

| Model | MAE | RMSE |
|---|---:|---:|
| Baseline | 143.64 | 506.74 |
| HistGradientBoosting | 145.27 | 347.94 |
| CatBoost | **112.80** | **303.97** |

**Conclusion:** CatBoost currently achieves the best results. Compared with the baseline, its MAE decreased by approximately 21.5% and its RMSE by approximately 40%.

In [16]:
from sklearn.metrics import mean_squared_log_error

def calculate_rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

In [17]:
# Calculate RMSLE for each model
baseline_rmsle = calculate_rmsle(
    y_valid, valid_prediction
)

hist_rmsle = calculate_rmsle(
    y_valid, prediction
)

cat_rmsle = calculate_rmsle(
    y_valid, cat_predictions
)

print(f'Baseline RMSLE:       {baseline_rmsle:.4f}')
print(f'HistGradient RMSLE:   {hist_rmsle:.4f}')
print(f'CatBoost RMSLE:       {cat_rmsle:.4f}')

Baseline RMSLE:       0.6957
HistGradient RMSLE:   1.6181
CatBoost RMSLE:       1.3702


In [18]:
# Transform sales using log
y_train_log = np.log1p(y_train_sample)

cat_model_log = CatBoostRegressor(
    iterations=500, 
    learning_rate=0.1, 
    depth=8, 
    loss_function='RMSE', 
    random_seed=42, 
    verbose=100, 
    allow_writing_files=False
)

# Train the model on log-transformed sales
cat_model_log.fit(
    X_train_cat, 
    y_train_log, 
    cat_features=categorial_features
)

# predict log transformed sales
log_prediction = cat_model_log.predict(X_valid_cat)

# Convert back
cat_predictions_log = np.maximum(
    np.expm1(log_prediction), 0
)

# Evaluate the model 

cat_log_rmsle = calculate_rmsle(
    y_valid, 
    cat_predictions_log
)

print(f'Baseline RMLSE: {baseline_rmsle:.4f}')
print(f'Original CatBoost RMSLE {cat_rmsle:.4f}')
print(f'Log-target CatBoost RMSLE: {cat_log_rmsle:.4}')

0:	learn: 2.5049059	total: 41.8ms	remaining: 20.8s
100:	learn: 0.8265208	total: 7.3s	remaining: 28.9s
200:	learn: 0.7110789	total: 15.3s	remaining: 22.8s
300:	learn: 0.6498787	total: 22.8s	remaining: 15.1s
400:	learn: 0.6141706	total: 31.9s	remaining: 7.87s
499:	learn: 0.5856866	total: 40.3s	remaining: 0us
Baseline RMLSE: 0.6957
Original CatBoost RMSLE 1.3702
Log-target CatBoost RMSLE: 0.5337


In [19]:
# Error analysis 
error_analysis = valid_data[
    ['date', 'store_nbr', 'family', 'sales']
].copy()

error_analysis['prediction'] = cat_predictions_log

error_analysis['error'] = (
    error_analysis['prediction'] - error_analysis['sales']
)

error_analysis['absolute_error'] = (
    error_analysis['error'].abs()
)

error_analysis.sort_values(
    'absolute_error', ascending=False
).head(10)

,date,store_nbr,family,sales,prediction,error,absolute_error
2996797,2017-08-13,44,GROCERY I,9811.000,17293.556987,7482.556987,7482.556987
2974896,2017-08-01,3,GROCERY I,8229.000,1349.680807,-6879.319193,6879.319193
2982642,2017-08-05,47,BEVERAGES,14469.000,7934.014621,-6534.985379,6534.985379
2995026,2017-08-12,44,GROCERY I,9391.000,15647.614600,6256.614600,6256.614600
2974887,2017-08-01,3,BEVERAGES,8192.000,1991.813966,-6200.186034,6200.186034
2984688,2017-08-06,54,BEVERAGES,8552.000,2538.341483,-6013.658517,6013.658517
2975128,2017-08-01,36,GROCERY I,6013.617,360.701308,-5652.915692,5652.915692
2980629,2017-08-04,40,BEVERAGES,8272.000,2696.914594,-5575.085406,5575.085406
2996214,2017-08-13,28,GROCERY I,11002.664,5540.446480,-5462.217520,5462.217520
2996973,2017-08-13,49,GROCERY I,7655.000,13078.095308,5423.095308,5423.095308


In [20]:
# Group predicted errors by product family
product_errors = (
    error_analysis
    .groupby('family')
    .agg(
        actual_sales=('sales', 'mean'),
        predicted_sales=('prediction', 'mean'),
        MAE=('absolute_error', 'mean'),
        obersvations=('sales', 'size')
    )
    .sort_values('MAE', ascending=False)
)

product_errors.head(10)

,actual_sales,predicted_sales,MAE,obersvations
family,,,,
BEVERAGES,3469.016049,2629.633312,909.972979,810
GROCERY I,4580.489899,4410.344365,896.989325,810
PRODUCE,2290.857758,2221.161580,547.259514,810
CLEANING,1204.762963,1111.920159,289.038087,810
BREAD/BAKERY,538.782429,416.094790,156.708733,810
DAIRY,832.766667,836.722438,153.063408,810
MEATS,374.035392,328.835931,106.716909,810
POULTRY,375.126746,308.655885,102.869069,810
DELI,302.746394,341.789977,95.593200,810


In [21]:
# Analyze pred errors by store and product family 
store_family_error = (
    error_analysis
    .groupby(['store_nbr', 'family'])
    .agg(
        actual_sales=('sales', 'mean'), 
        predicted_sales=('prediction', 'mean'), 
        MAE=('absolute_error', 'mean'),
        observations=('sales', 'size')
    )
    .reset_index()
    .sort_values('MAE', ascending=False)
)

store_family_error.head(10)

,store_nbr,family,actual_sales,predicted_sales,MAE,observations
1713,52,PRODUCE,4945.064533,1716.513223,3228.551310,15
1431,44,GROCERY I,9536.800000,12388.166973,2851.366973,15
1389,43,BEVERAGES,5291.666667,2574.556752,2717.109915,15
69,3,BEVERAGES,7878.466667,5188.219002,2690.247665,15
1521,47,BEVERAGES,8550.666667,5965.199327,2585.467340,15
1290,40,BEVERAGES,5371.000000,2952.836466,2418.163534,15
1455,45,BEVERAGES,8726.733333,6313.000898,2413.732435,15
639,20,GROCERY I,6512.872533,4328.260659,2184.611874,15
1752,54,BEVERAGES,3933.466667,1936.746260,2047.786335,15
342,11,GROCERY I,7950.727600,6168.791419,1988.017469,15


In [23]:
# Analyze pred errors by promotion stats
promotion_errors = error_analysis.copy()

promotion_errors['onpromotion'] = valid_data['onpromotion'].values
promotion_summary = (
    promotion_errors
    .groupby('onpromotion')
    .agg(
        actual_sales=('sales', 'mean'),
        predicted_sales=('prediction', 'mean'), 
        MAE=('absolute_error', 'mean'),
        observations=('sales', 'size')
    )
    .sort_values('MAE', ascending=False)
)

promotion_summary

,actual_sales,predicted_sales,MAE,observations
onpromotion,,,,
207,8154.466000,3644.736330,4509.729670,1
209,7873.919000,3489.986355,4383.932645,1
214,2940.584000,6217.093254,3276.509254,1
194,1985.897000,4301.337387,2315.440387,1
233,7743.166000,10013.737755,2270.571755,1
...,...,...,...,...
103,481.000000,509.817087,28.817087,1
158,2541.333000,2518.221229,23.111771,1
0,54.323631,45.560489,19.617084,14946


### Prediction Error Analysis

The prediction error analysis showed that the largest errors occurred mainly in the product families **BEVERAGES**, **GROCERY I**, and **PRODUCE**.

- The model both overestimated and underestimated sales.
- Some store-family combinations had particularly large prediction errors.
- Grouping errors by product family helped identify where predictions were less accurate.

**Conclusion:** Error analysis provides useful insights into model performance. However, further detailed investigations were postponed to keep the project focused on improving RMSLE and completing the Kaggle submission.

In [25]:
y_valid_log = np.log1p(y_valid)

cat_model_tuned = CatBoostRegressor(
    iterations=1000, 
    learning_rate=0.1, 
    depth=8, 
    loss_function='RMSE', 
    eval_metric='RMSE',
    random_seed=42, 
    verbose=100, 
    allow_writing_files=False
)

cat_model_tuned.fit(
    X_train_cat, 
    y_train_log, 
    cat_features=categorial_features, 
    eval_set=(X_valid_cat, y_valid_log), 
    early_stopping_rounds=100, 
    use_best_model=True
)

tuned_log_prediction = cat_model_tuned.predict(X_valid_cat)

tuned_prediction = np.maximum(
    np.expm1(tuned_log_prediction), 0
)

# Evaluate the model
tuned_rmsle = calculate_rmsle(
    y_valid, 
    tuned_prediction
)

print(f'Previos RMSLE: {cat_log_rmsle:.4f}')
print(f'Tuned RMSLE: {tuned_rmsle:.4f}')
print(f'Best Iteration: {cat_model_tuned.best_iteration_}')

0:	learn: 2.5049059	test: 2.4046045	best: 2.4046045 (0)	total: 44.9ms	remaining: 44.9s
100:	learn: 0.8265208	test: 0.7041368	best: 0.7041368 (100)	total: 7.14s	remaining: 1m 3s
200:	learn: 0.7110789	test: 0.6132232	best: 0.6132232 (200)	total: 15s	remaining: 59.4s
300:	learn: 0.6498787	test: 0.5745205	best: 0.5745205 (300)	total: 22.6s	remaining: 52.4s
400:	learn: 0.6141706	test: 0.5460532	best: 0.5460532 (400)	total: 30.9s	remaining: 46.2s
500:	learn: 0.5852692	test: 0.5339481	best: 0.5339481 (500)	total: 39.1s	remaining: 39s
600:	learn: 0.5658090	test: 0.5246308	best: 0.5246308 (600)	total: 47.5s	remaining: 31.5s
700:	learn: 0.5510937	test: 0.5168833	best: 0.5166832 (690)	total: 57.3s	remaining: 24.4s
800:	learn: 0.5381935	test: 0.5105532	best: 0.5105532 (800)	total: 1m 5s	remaining: 16.3s
900:	learn: 0.5271444	test: 0.5031548	best: 0.5031548 (900)	total: 1m 13s	remaining: 8.05s
999:	learn: 0.5185823	test: 0.4966107	best: 0.4966107 (999)	total: 1m 21s	remaining: 0us

bestTest = 0.496

In [34]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

X_train_lgb = X_train_cat.copy()
X_valid_lgb = X_valid_cat.copy()

# Textspalten in kategoriale Datentypen umwandeln
for col in categorial_features:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_valid_lgb[col] = X_valid_lgb[col].astype('category')

lgb_model = LGBMRegressor(
    n_estimators=1500, 
    learning_rate=0.05, 
    num_leaves=31, 
    max_depth=-1, 
    random_state=42, 
    verbosity=1
)

lgb_model.fit(
    X_train_lgb, 
    y_train_log, 
    eval_set=[(X_valid_lgb, y_valid_log)], 
    callbacks=[
        early_stopping(stopping_rounds=100), 
        log_evaluation(period=200)
    ], 
    categorical_feature=categorial_features
)

lgb_log_prediction = lgb_model.predict(X_valid_lgb)

lgb_prediction = np.maximum(
    np.expm1(lgb_log_prediction), 0
)

lgb_rmsle = calculate_rmsle(y_valid, lgb_prediction)

print(f'CatBoost RMSLE: {tuned_rmsle:.4f}')
print(f'LightGBM RMSLE {lgb_rmsle:.4f}')
print(f'Best Iteration: {lgb_model.best_iteration_}')

/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005460 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 929
[LightGBM] [Info] Number of data points in the train set: 300000, number of used features: 16
[LightGBM] [Info] Start training from score 2.919879
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l2: 0.35962
[400]	valid_0's l2: 0.270208
[600]	valid_0's l2: 0.250747
[800]	valid_0's l2: 0.240022
[1000]	valid_0's l2: 0.230958
[1200]	valid_0's l2: 0.223331
[1400]	valid_0's l2: 0.22054
Did not meet early stopping. Best iteration is:
[1499]	valid_0's l2: 0.218508
CatBoost RMSLE: 0.4963
LightGBM RMSLE 0.4674
Best Iteration: 1499
